# Temporary Google Colab PyTorch / GPU Showcase

Open this notebook in Colab, choose a GPU runtime if available, and select **Run all**. The setup cell tries the JSPCV attachment manifest and falls back to the same public GitHub files while this temporary local demonstration is being reviewed.

In [ ]:
import json
from pathlib import Path
from urllib.request import Request, urlopen

MANIFEST_URL = "https://jspcv.com/resources/colabAttachments/temporary-colab-gpu-showcase.ipynb/manifest.json"
FALLBACK_FILES = {
    "temporary-demo-data.csv": "https://raw.githubusercontent.com/sonamu-jun/jspcv-colab-resources/demo-showcase-20260813/demo_showcase/temporary-demo-data.csv",
    "temporary-demo-config.json": "https://raw.githubusercontent.com/sonamu-jun/jspcv-colab-resources/demo-showcase-20260813/demo_showcase/temporary-demo-config.json",
}

try:
    request = Request(MANIFEST_URL, headers={"User-Agent": "jspcv-temporary-showcase"})
    with urlopen(request, timeout=15) as response:
        files = {item["name"]: item["url"] for item in json.load(response)["attachments"]}
    source = "JSPCV manifest"
except Exception:
    files = FALLBACK_FILES
    source = "temporary GitHub fallback"

data_dir = Path("attachments")
data_dir.mkdir(exist_ok=True)
for name, url in files.items():
    with urlopen(Request(url, headers={"User-Agent": "jspcv-temporary-showcase"}), timeout=30) as response:
        (data_dir / name).write_bytes(response.read())
print(f"Loaded {len(files)} attachments from {source}: {sorted(files)}")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Runtime device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not enabled. In Colab, choose Runtime > Change runtime type > GPU.")

In [ ]:
import pandas as pd

frame = pd.read_csv(data_dir / "temporary-demo-data.csv")
x = torch.arange(len(frame), dtype=torch.float32, device=device).reshape(-1, 1)
y = torch.tensor(frame["score"].to_numpy(), dtype=torch.float32, device=device).reshape(-1, 1)
model = torch.nn.Linear(1, 1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()

loss_history = []
for _ in range(250):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach().cpu()))

print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.plot(loss_history, color="#0969da")
plt.yscale("log")
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title(f"PyTorch training on {device}")
plt.tight_layout()
plt.show()